# Zomathon Problem Statement: Intelligent Cart Add-On Recommender

**Objective:** Build a machine learning recommendation engine that suggests relevant add-on items (desserts, beverages, meal complements) to users at checkout.

**Business Goal:** Increase Average Order Value (AOV) by predicting high-conversion add-ons, while minimizing user friction to prevent cart abandonment.

This notebook is divided into two phases:
1. **Data Pipeline Simulation:** Generating a realistic, noisy dataset mimicking live user telemetry.
2. **Predictive Modeling:** Engineering domain-specific features and training a LightGBM ranking classifier, evaluated via strict offline metrics (AUC, Precision@K, NDCG).

In [ ]:
# Install necessary libraries for the environment
!pip install pandas numpy lightgbm scikit-learn

## Phase 1: Data Generation & Pipeline Simulation
Real-world data is rarely clean. To properly test our architecture, we simulate a messy database containing realistic user behaviors, meal times, and dietary preferences. We also intentionally inject missing values (NaNs) and extreme outliers to ensure our model handles production-level noise.

In [ ]:
import pandas as pd
import numpy as np
import random

# Set random seed for reproducibility
np.random.seed(42)
num_rows = 2000

# Helper lists
main_items = ['Chicken Biryani', 'Margherita Pizza', 'Masala Dosa', 'Beef Burger', 'Paneer Butter Masala', 'Sushi Platter', 'Oatmeal Bowl', 'Pasta Alfredo']
addon_items = ['Mint Raita', 'Garlic Bread', 'Filter Coffee', 'French Fries', 'Butter Naan', 'Miso Soup', 'Green Tea', 'Chocolate Brownie', 'Extra Chutney', 'Healthy Quinoa Salad']
diet_trends = ['Junk', 'Healthy', 'Balanced']
dietary_types = ['Veg', 'Non-Veg', 'Vegan']

# Simulate realistic ordering hours (weighted towards traditional meal times)
hours_distribution = [
    0, 1, 2,               # Late night
    7, 8, 9, 10,           # Breakfast
    12, 13, 14, 15,        # Lunch
    19, 20, 21, 22, 23     # Dinner
]

# Generate base data
data = {
    'order_id': [f"ORD{10000 + i}" for i in range(num_rows)],
    'user_id': [f"U{random.randint(1000, 1500)}" for _ in range(num_rows)],
    'restaurant_id': [f"R{random.randint(500, 550)}" for _ in range(num_rows)],
    'current_cart_items': np.random.choice(main_items, num_rows),
    'candidate_addon_item': np.random.choice(addon_items, num_rows),
    'is_meal_complement': np.random.choice([0, 1], num_rows, p=[0.4, 0.6]),
    'user_item_frequency': np.random.poisson(lam=3, size=num_rows),
    'user_recent_diet_trend': np.random.choice(diet_trends, num_rows),
    'addon_dietary_type': np.random.choice(dietary_types, num_rows),
    'cart_dietary_type': np.random.choice(dietary_types, num_rows),
    'delivery_distance_km': np.round(np.random.uniform(0.5, 15.0, num_rows), 1),
    'order_hour_of_day': np.random.choice(hours_distribution, num_rows), # NUMERIC TIME COLUMN (0-23)
    'addon_price': np.round(np.random.uniform(20, 250, num_rows), 0),
    'cart_value': np.round(np.random.uniform(150, 1500, num_rows), 0),
    'item_rating': np.round(np.random.uniform(3.0, 5.0, num_rows), 1),
    'is_top_seller': np.random.choice([0, 1], num_rows, p=[0.7, 0.3]),
    'target_added_to_cart': np.random.choice([0, 1], num_rows, p=[0.8, 0.2])
}

df = pd.DataFrame(data)

# --- INJECT OUTLIERS ---
outlier_indices_cart = np.random.choice(df.index, size=20, replace=False)
df.loc[outlier_indices_cart, 'cart_value'] = np.random.uniform(5000, 25000, 20)

outlier_indices_dist = np.random.choice(df.index, size=15, replace=False)
df.loc[outlier_indices_dist, 'delivery_distance_km'] = np.random.uniform(50, 250, 15)

outlier_indices_price = np.random.choice(df.index, size=5, replace=False)
df.loc[outlier_indices_price, 'addon_price'] = -50

# --- INJECT MISSING VALUES (NaNs) ---
missing_rating_idx = np.random.choice(df.index, size=150, replace=False)
df.loc[missing_rating_idx, 'item_rating'] = np.nan

missing_diet_idx = np.random.choice(df.index, size=80, replace=False)
df.loc[missing_diet_idx, 'user_recent_diet_trend'] = np.nan

missing_dist_idx = np.random.choice(df.index, size=40, replace=False)
df.loc[missing_dist_idx, 'delivery_distance_km'] = np.nan

# Save to CSV
file_name = 'zomathon_ps2_final_dataset.csv'
df.to_csv(file_name, index=False)

print(f"Dataset saved as '{file_name}' with numeric time hours.")

Dataset saved as 'zomathon_ps2_final_dataset.csv' with numeric time hours.


## Phase 2: Feature Engineering & LightGBM Model
We build our recommendation logic using LightGBM, avoiding data leakage through a strict temporal split.

**Key Implementations:**
* **Custom Features:** Engineered `Price_Ratio`, `Dietary_Match`, and `Distance_x_Hour` to capture true purchasing behavior rather than relying purely on raw columns.
* **Temporal Split:** Data is sorted chronologically by `order_id` to train on the past and predict the future.
* **Evaluation:** Scored using production-grade ranking metrics (`NDCG`, `Precision@K`) alongside segmented error analysis to track bias across High vs. Low value carts.

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, ndcg_score
import warnings

warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# 1. Load Data & Feature Engineering
# ---------------------------------------------------------
df = pd.read_csv('zomathon_ps2_final_dataset.csv')

# Requirement 1a: Temporal Split (simulated by sorting order_id)
df = df.sort_values('order_id')

# Injecting Business Logic Features
df['Price_Ratio'] = df['addon_price'] / (df['cart_value'] + 1)
df['Dietary_Match'] = (df['addon_dietary_type'] == df['cart_dietary_type']).astype(int)
df['Distance_x_Hour'] = df['delivery_distance_km'] * df['order_hour_of_day']

# ---------------------------------------------------------
# 2. Split and Setup
# ---------------------------------------------------------
# Dropping user_id and restaurant_id to prevent memorization/overfitting on small datasets
features = [
    'candidate_addon_item', 'is_meal_complement',
    'user_item_frequency', 'delivery_distance_km', 'order_hour_of_day',
    'addon_price', 'cart_value', 'item_rating', 'is_top_seller',
    'addon_dietary_type', 'cart_dietary_type',
    'Price_Ratio', 'Dietary_Match', 'Distance_x_Hour'
]

categorical_cols = ['candidate_addon_item', 'addon_dietary_type', 'cart_dietary_type']
target = 'target_added_to_cart'

split_index = int(len(df) * 0.8)
train = df.iloc[:split_index].copy()
test = df.iloc[split_index:].copy()

for col in categorical_cols:
    train[col] = train[col].astype('category')
    test[col] = test[col].astype('category')

# ---------------------------------------------------------
# 3. Train LightGBM
# ---------------------------------------------------------
model = lgb.LGBMClassifier(
    random_state=42,
    n_estimators=100,
    max_depth=4,
    learning_rate=0.05,
    min_child_samples=40,
    is_unbalance=True,      # Handles class imbalance ethically
    verbose=-1
)

model.fit(train[features], train[target])

# ---------------------------------------------------------
# 4. Offline Evaluation
# ---------------------------------------------------------
test['Predicted_Score'] = model.predict_proba(test[features])[:, 1]

print("--- Global Performance ---")
global_auc = roc_auc_score(test[target], test['Predicted_Score'])
print(f"Overall AUC: {global_auc:.4f}\n")

print("--- Ranking Metrics @ K=3 ---")
K = 3

def calculate_user_ranking_metrics(user_group):
    if len(user_group) < 2 or user_group[target].sum() == 0:
        return pd.Series({'Precision@K': np.nan, 'Recall@K': np.nan, 'NDCG': np.nan})

    y_true = user_group[target].values
    y_score = user_group['Predicted_Score'].values

    top_k_idx = np.argsort(y_score)[::-1][:K]
    top_k_true = y_true[top_k_idx]

    precision = top_k_true.sum() / K
    recall = top_k_true.sum() / y_true.sum()

    try:
        ndcg = ndcg_score([y_true], [y_score], k=K)
    except ValueError:
        ndcg = np.nan

    return pd.Series({'Precision@K': precision, 'Recall@K': recall, 'NDCG': ndcg})

user_metrics_df = test.groupby('user_id').apply(calculate_user_ranking_metrics).dropna()
print(user_metrics_df.mean().round(4).to_string())

# Error Analysis by Segment
print("\n--- Error Analysis by Cart Value Segment ---")
test['Cart_Value_Segment'] = pd.qcut(test['cart_value'], q=2, labels=['Low Value', 'High Value'])

for segment, group in test.groupby('Cart_Value_Segment', observed=False):
    if len(group[target].unique()) > 1:
        seg_auc = roc_auc_score(group[target], group['Predicted_Score'])
        print(f"{segment} AUC: {seg_auc:.4f}")

--- Global Performance ---
Overall AUC: 0.5561

--- Ranking Metrics @ K=3 ---
Precision@K    0.3509
Recall@K       0.9737
NDCG           0.7940

--- Error Analysis by Cart Value Segment ---
Low Value AUC: 0.5545
High Value AUC: 0.5593
